# EDA อัตราการต่อสัญญาเช่า (Renewal Rate)

Notebook นี้ใช้สำรวจและพยากรณ์ **อัตราการต่อสัญญาเช่ารายเดือนในระดับโครงการ** โดยหนึ่งแถวแทนหนึ่งโครงการในหนึ่งเดือน

เป้าหมายหลัก:
- สร้างข้อมูลระดับโครงการ–เดือนจากข้อมูลสัญญาเช่ารายสัญญา
- ศึกษาการกระจาย แนวโน้ม และความแตกต่างของอัตราการต่อสัญญา
- ตรวจสอบ Historical และ Lag Features โดยไม่ใช้ข้อมูลอนาคต
- เปรียบเทียบ Historical-rate Baseline กับ Binomial Logistic Regression
- ประเมินด้วย Rolling Backtest, MAE, RMSE และ Binomial Log Loss


## 1. Import เครื่องมือ

- `pandas` และ `numpy` ใช้จัดการข้อมูลและคำนวณค่าสถิติ
- `matplotlib` และ `seaborn` ใช้สร้างกราฟ
- ฟังก์ชันใน `kaverentai` ใช้สร้างข้อมูลและโหลด Feature ที่ทีมเตรียมไว้


In [ ]:
from pathlib import Path
import json
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from kaverentai.data.load_data import load_all_data
from kaverentai.features.renewal_rate_features import (
    RATE_MODEL_FEATURES,
    build_renewal_rate_data,
)

DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.options.display.float_format = "{:,.4f}".format
PROJECT_ROOT


## 2. โหลดข้อมูล

โหลดตารางสัญญาและสภาพตลาดจาก `data/processed/` แล้วสร้างข้อมูลโครงการ–เดือนโดยตรงด้วยฟังก์ชันเดียวกับที่ใช้ฝึกโมเดล เพื่อให้ EDA และโมเดลใช้กติกาเดียวกัน

ตารางสำคัญที่ใช้:
- `leases` ใช้สร้างจำนวนสัญญาหมดอายุและจำนวนที่ต่อ
- `weekly_market` ใช้ระบุวันที่หมดอายุและสภาพตลาดย้อนหลัง
- ผลลัพธ์สุดท้ายมีหนึ่งแถวต่อหนึ่งโครงการ–เดือน


In [ ]:
tables = load_all_data(cleaned=True)
leases = tables["leases"].copy()
weekly_market = tables["weekly_market"].copy()
renewal_rate_data = build_renewal_rate_data(
    leases=leases,
    weekly_market=weekly_market,
)

print("ข้อมูลสัญญาต้นทาง:", leases.shape)
print("ข้อมูลตลาดต้นทาง:", weekly_market.shape)
print("ข้อมูลระดับโครงการ-เดือน:", renewal_rate_data.shape)


### ดูตัวอย่างข้อมูลที่ใช้วิเคราะห์

หนึ่งแถวด้านล่างหมายถึงอัตราการต่อสัญญาของหนึ่งโครงการในหนึ่งเดือน โดย `renewed_contracts / expiring_contracts` เท่ากับ `renewal_rate`


In [ ]:
renewal_rate_data.head()


## 3. กำหนดหน่วยวิเคราะห์และ Target

Target ใหม่คือ `renewal_rate` ซึ่งมีค่าระหว่าง 0–1 ไม่ใช่คำตอบรายสัญญาแบบ 0/1

ตัวอย่าง: ถ้าโครงการหนึ่งมีสัญญาหมดอายุ 100 ฉบับและต่อ 48 ฉบับ อัตราการต่อสัญญาของโครงการ–เดือนนั้นเท่ากับ 48%


In [ ]:
target_check = renewal_rate_data.assign(
    calculated_rate=(
        renewal_rate_data["renewed_contracts"]
        / renewal_rate_data["expiring_contracts"]
    )
)[["project_id", "expiry_month", "renewed_contracts",
   "expiring_contracts", "renewal_rate", "calculated_rate"]]

target_check.head()


### แบ่งข้อมูลก่อนทำ EDA

การสำรวจความสัมพันธ์และเลือก Feature ใช้เฉพาะ `train` เพื่อไม่ให้เห็นข้อมูล Validation และ Test ล่วงหน้า ส่วน Validation จะเปิดเมื่อประเมินโมเดลเท่านั้น


In [ ]:
eda_data = renewal_rate_data.loc[
    renewal_rate_data["data_split"].eq("train")
].copy()

print("จำนวนแถวสำหรับ EDA:", len(eda_data))
print(
    "ช่วงเวลา:",
    eda_data["expiry_month"].min().strftime("%Y-%m"),
    "ถึง",
    eda_data["expiry_month"].max().strftime("%Y-%m"),
)


## 4. ตรวจสอบคุณภาพข้อมูลเบื้องต้น

ตรวจจำนวนแถว ชนิดข้อมูล Missing Value และความซ้ำของคีย์ `project_id + expiry_month` ก่อนเริ่มวิเคราะห์


In [ ]:
print("จำนวนแถว:", renewal_rate_data.shape[0])
print("จำนวนคอลัมน์:", renewal_rate_data.shape[1])
renewal_rate_data.info()


In [ ]:
missing_summary = (
    renewal_rate_data.isna().sum()
    .rename("missing")
    .to_frame()
)
missing_summary["missing_pct"] = (
    missing_summary["missing"] / len(renewal_rate_data) * 100
)
missing_summary.loc[missing_summary["missing"].gt(0)]


In [ ]:
duplicate_count = renewal_rate_data.duplicated(
    ["project_id", "expiry_month"]
).sum()
invalid_rate_count = (~renewal_rate_data["renewal_rate"].between(0, 1)).sum()

print("โครงการ-เดือนที่ซ้ำ:", duplicate_count)
print("อัตราต่อสัญญาที่อยู่นอกช่วง 0-1:", invalid_rate_count)


Missing Value ใน Historical และ Lag Features ของเดือนแรกแต่ละโครงการเป็นสิ่งที่คาดไว้ เพราะยังไม่มีเดือนก่อนหน้าให้คำนวณ โมเดลจะเติมค่ากลางจาก Train ผ่าน Pipeline โดยไม่แก้ข้อมูลต้นฉบับ


## 5. สำรวจอัตราการต่อสัญญา

เริ่มจากดูอัตรารวม การกระจายของอัตรารายโครงการ–เดือน และจำนวนสัญญาที่หมดอายุในแต่ละแถว

ต้องดูจำนวนสัญญาควบคู่กับอัตราเสมอ เพราะอัตราจากโครงการ–เดือนที่มีสัญญาน้อยจะผันผวนมากกว่า


In [ ]:
train_overall_rate = (
    eda_data["renewed_contracts"].sum()
    / eda_data["expiring_contracts"].sum()
)

print(f"อัตราต่อสัญญารวมใน Train: {train_overall_rate:.2%}")
display(eda_data["renewal_rate"].describe())
display(eda_data["expiring_contracts"].describe())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(
    data=eda_data, x="renewal_rate", bins=20, kde=True, ax=axes[0]
)
axes[0].axvline(train_overall_rate, color="red", linestyle="--")
axes[0].set_title("Distribution of Project-Month Renewal Rate")
axes[0].set_xlabel("Renewal Rate")

sns.histplot(
    data=eda_data, x="expiring_contracts", bins=20, ax=axes[1]
)
axes[1].set_title("Number of Expiring Contracts per Project-Month")
axes[1].set_xlabel("Expiring Contracts")

plt.tight_layout()
plt.show()


### Correlation Matrix

ใช้ตรวจความสัมพันธ์เชิงเส้นเบื้องต้นระหว่าง `renewal_rate` กับตัวแปรเชิงตัวเลข โดยยังไม่สรุปว่าเป็นเหตุและผล


In [ ]:
correlation_columns = [
    "renewal_rate",
    "expiring_contracts",
    "share_current_renewal",
    "share_12_month_contract",
    "mean_prior_renewal_rate",
    "project_previous_observed_rate",
    "project_historical_rate",
    "overall_historical_rate",
    "lagged_rent_to_market_ratio",
    "lagged_occupancy",
    "lagged_demand_pressure",
]

plt.figure(figsize=(12, 8))
sns.heatmap(
    eda_data[correlation_columns].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
)
plt.title("Correlation Matrix: Train Project-Month Data")
plt.tight_layout()
plt.show()


กราฟและ Correlation Matrix ใช้ดูว่ามีสัญญาณที่เสถียรหรือไม่ หากความสัมพันธ์กับ Target ต่ำ การใช้แบบจำลองที่ซับซ้อนอาจไม่ช่วย และ Historical-rate Baseline อาจเหมาะสมกว่า


## 6. เปรียบเทียบอัตราต่อสัญญาระหว่างโครงการ

คำนวณอัตราต่อสัญญาแบบถ่วงน้ำหนัก โดยรวมจำนวนสัญญาที่ต่อแล้วหารด้วยจำนวนสัญญาหมดอายุ ไม่ใช้ค่าเฉลี่ยธรรมดาของเปอร์เซ็นต์รายเดือน


In [ ]:
project_summary = (
    eda_data.groupby("project_id", as_index=False)
    .agg(
        renewed_contracts=("renewed_contracts", "sum"),
        expiring_contracts=("expiring_contracts", "sum"),
        observed_months=("expiry_month", "nunique"),
    )
)
project_summary["renewal_rate"] = (
    project_summary["renewed_contracts"]
    / project_summary["expiring_contracts"]
)
project_summary = project_summary.sort_values("renewal_rate")
project_summary


In [ ]:
plt.figure(figsize=(11, 6))
sns.barplot(
    data=project_summary,
    x="renewal_rate",
    y=project_summary["project_id"].astype(str),
    color="#457b9d",
)
plt.axvline(train_overall_rate, color="red", linestyle="--", label="Train overall")
plt.title("Weighted Renewal Rate by Project")
plt.xlabel("Renewal Rate")
plt.ylabel("Project ID")
plt.legend()
plt.tight_layout()
plt.show()


ควรพิจารณา `expiring_contracts` และ `observed_months` ร่วมด้วย โครงการที่มีข้อมูลน้อยอาจมีอัตราสูงหรือต่ำจากความผันผวนแบบสุ่ม ไม่ได้หมายความว่าโครงการนั้นบริหารผู้เช่าดีกว่าหรือแย่กว่าโดยตรง


## 7. ดูแนวโน้มอัตราต่อสัญญาตามเวลา

รวมผลทุกโครงการเป็นรายเดือนเพื่อดูแนวโน้มโดยรวมและจำนวนสัญญาที่หมดอายุในแต่ละเดือน


In [ ]:
monthly_summary = (
    eda_data.groupby("expiry_month", as_index=False)
    .agg(
        renewed_contracts=("renewed_contracts", "sum"),
        expiring_contracts=("expiring_contracts", "sum"),
    )
)
monthly_summary["renewal_rate"] = (
    monthly_summary["renewed_contracts"]
    / monthly_summary["expiring_contracts"]
)

plt.figure(figsize=(12, 5))
sns.lineplot(
    data=monthly_summary, x="expiry_month", y="renewal_rate", marker="o"
)
plt.axhline(train_overall_rate, color="red", linestyle="--", label="Train overall")
plt.title("Monthly Renewal Rate Across All Projects")
plt.xlabel("Expiry Month")
plt.ylabel("Renewal Rate")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 4))
sns.barplot(
    data=monthly_summary,
    x="expiry_month",
    y="expiring_contracts",
    color="#a8dadc",
)
plt.title("Monthly Expiring Contract Volume")
plt.xlabel("Expiry Month")
plt.ylabel("Expiring Contracts")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


ถ้าอัตรารายเดือนแกว่งรอบค่าเฉลี่ยโดยไม่มีแนวโน้มชัดเจน Baseline ที่ใช้อัตราเฉลี่ยในอดีตจะมีความเสถียร ส่วนเดือนที่มีสัญญาน้อยควรมีช่วงความไม่แน่นอนกว้างกว่า


## 8. วิเคราะห์ Historical และ Lag Features

`project_historical_rate` คำนวณจากผลลัพธ์ของเดือนก่อนหน้าเท่านั้น ส่วนตัวแปรตลาดใช้ค่าจากโครงการ–เดือนก่อนหน้า จึงไม่ดึงคำตอบของเดือนปัจจุบันมาใช้เป็น Feature


In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=eda_data,
    x="project_historical_rate",
    y="renewal_rate",
    size="expiring_contracts",
    sizes=(20, 220),
    alpha=0.65,
)
plt.axhline(train_overall_rate, color="red", linestyle="--")
plt.title("Historical Project Rate vs Current Renewal Rate")
plt.xlabel("Project Historical Renewal Rate")
plt.ylabel("Current Renewal Rate")
plt.tight_layout()
plt.show()


In [ ]:
lag_columns = [
    "project_previous_observed_rate",
    "project_historical_rate",
    "overall_historical_rate",
    "lagged_rent_to_market_ratio",
    "lagged_occupancy",
    "lagged_demand_pressure",
]

lag_correlations = (
    eda_data[["renewal_rate", *lag_columns]]
    .corr()[["renewal_rate"]]
    .drop(index="renewal_rate")
    .sort_values("renewal_rate", key=abs, ascending=False)
)
lag_correlations


Historical และ Lag Features มีไว้ให้โมเดลปรับค่าตามโครงการและเวลา แต่ถ้าความสัมพันธ์ยังต่ำ โมเดล Logistic ที่มีหลายตัวแปรอาจเพิ่มความผันผวนมากกว่าลดข้อผิดพลาด


## 9. ตรวจสอบการแบ่งข้อมูลตามเวลา

- Train: ถึงเดือนธันวาคม 2025
- Validation: มกราคม–มีนาคม 2026
- Test: เมษายน–มิถุนายน 2026

การเลือก Feature และเปรียบเทียบโมเดลใน Notebook นี้ไม่ใช้ Test ซ้ำ


In [ ]:
split_summary = (
    renewal_rate_data.groupby("data_split", as_index=False)
    .agg(
        project_month_rows=("renewal_rate", "size"),
        contracts=("expiring_contracts", "sum"),
        first_month=("expiry_month", "min"),
        last_month=("expiry_month", "max"),
    )
)
split_summary


In [ ]:
split_months = renewal_rate_data[["expiry_month", "data_split"]].drop_duplicates()
plt.figure(figsize=(12, 2.5))
sns.scatterplot(
    data=split_months,
    x="expiry_month",
    y=np.ones(len(split_months)),
    hue="data_split",
    s=90,
)
plt.title("Chronological Data Split")
plt.xlabel("Expiry Month")
plt.yticks([])
plt.tight_layout()
plt.show()


## 10. สรุปผล EDA

ประเด็นที่ควรนำไปเขียนรายงาน:

1. หน่วยวิเคราะห์เปลี่ยนเป็นโครงการ–เดือน และ Target คืออัตราการต่อสัญญา
2. ต้องถ่วงน้ำหนักตามจำนวนสัญญาหมดอายุ เพราะแต่ละแถวมีจำนวนสัญญาไม่เท่ากัน
3. Missing Value ของ Lag Features ในเดือนแรกเป็นสิ่งที่เกิดตามธรรมชาติ
4. Feature ทุกตัวต้องรู้ได้ก่อนผลการต่อสัญญาของเดือนปัจจุบัน
5. หากความสัมพันธ์ระหว่าง Feature กับ Target ต่ำ ควรให้ความสำคัญกับ Baseline ที่เรียบง่าย


In [ ]:
eda_key_numbers = pd.Series({
    "project_month_rows": len(renewal_rate_data),
    "projects": renewal_rate_data["project_id"].nunique(),
    "months": renewal_rate_data["expiry_month"].nunique(),
    "contracts": renewal_rate_data["expiring_contracts"].sum(),
    "renewed_contracts": renewal_rate_data["renewed_contracts"].sum(),
    "train_overall_rate": train_overall_rate,
})
eda_key_numbers


# ส่วนที่ 2: การฝึกและประเมินโมเดลพยากรณ์อัตราต่อสัญญา

เปรียบเทียบ 2 วิธีทางสถิติ:

1. **Historical-rate Baseline** ใช้อัตราต่อสัญญาถ่วงน้ำหนักจากข้อมูลในอดีต
2. **Binomial Logistic Regression** เรียนรู้จาก Feature ระดับโครงการ–เดือน โดยให้น้ำหนักตามจำนวนสัญญา


## 11. โหลดผลการฝึกโมเดล

โหลด Metrics ผลทำนาย Validation ผล Rolling Backtest และโมเดลที่ฝึกจากสคริปต์ `train_renewal_rate.py`


In [ ]:
metrics_path = REPORT_DIR / "metrics" / "renewal_rate_metrics.json"
validation_path = REPORT_DIR / "tables" / "renewal_rate_validation_predictions.csv"
backtest_path = REPORT_DIR / "tables" / "renewal_rate_backtest_predictions.csv"
tuning_path = REPORT_DIR / "tables" / "renewal_rate_logistic_tuning.csv"
test_path = REPORT_DIR / "tables" / "renewal_rate_test_predictions.csv"
business_path = REPORT_DIR / "tables" / "renewal_rate_business_predictions.csv"
model_path = MODEL_DIR / "renewal_rate_logistic.joblib"

metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
validation_predictions = pd.read_csv(validation_path, parse_dates=["expiry_month"])
backtest_predictions = pd.read_csv(backtest_path, parse_dates=["expiry_month"])
tuning_results = pd.read_csv(tuning_path)
test_predictions = pd.read_csv(test_path, parse_dates=["expiry_month"])
business_predictions = pd.read_csv(business_path, parse_dates=["expiry_month"])
rate_model = joblib.load(model_path)

print("Validation rows:", len(validation_predictions))
print("Backtest rows:", len(backtest_predictions))
print("Tuning candidates:", len(tuning_results))
print("Test rows:", len(test_predictions))
print("Business output rows:", len(business_predictions))


## 12. Rolling Time Validation

ใช้ Expanding Window 4 รอบภายในปี 2025 โดยฝึกจากอดีตและตรวจสอบกับไตรมาสถัดไป ช่วยประเมินความเสถียรของโมเดลในหลายช่วงเวลา

- **MAE** คือความคลาดเคลื่อนสัมบูรณ์เฉลี่ย ค่ายิ่งต่ำยิ่งดี
- **RMSE** ให้น้ำหนักกับความผิดพลาดขนาดใหญ่มากขึ้น ค่ายิ่งต่ำยิ่งดี
- **Binomial Log Loss** ประเมินคุณภาพของความน่าจะเป็น ค่ายิ่งต่ำยิ่งดี


In [ ]:
backtest_metrics = (
    pd.DataFrame(metrics["rolling_backtest_2025"])
    .T.reset_index(names="model")
)
backtest_metrics[[
    "model", "weighted_mae", "weighted_rmse",
    "binomial_log_loss", "weighted_bias", "contracts"
]]


In [ ]:
fold_rows = []
for fold, fold_data in backtest_predictions.groupby("fold"):
    weights = fold_data["expiring_contracts"]
    for model_name, prediction_column in {
        "Historical Baseline": "baseline_predicted_rate",
        "Binomial Logistic": "logistic_predicted_rate",
    }.items():
        fold_rows.append({
            "fold": fold,
            "model": model_name,
            "weighted_mae": np.average(
                abs(fold_data[prediction_column] - fold_data["renewal_rate"]),
                weights=weights,
            ),
        })

fold_metrics = pd.DataFrame(fold_rows)
plt.figure(figsize=(10, 5))
sns.barplot(data=fold_metrics, x="fold", y="weighted_mae", hue="model")
plt.title("Weighted MAE by Rolling Backtest Fold")
plt.xlabel("Fold")
plt.ylabel("Weighted MAE")
plt.tight_layout()
plt.show()


ผล Rolling Backtest แสดงว่า Historical-rate Baseline มี MAE ประมาณ 3.9 จุดเปอร์เซ็นต์ และมีความเสถียรกว่า Binomial Logistic Regression ในข้อมูลชุดนี้


## 13. ผลการ Tune Binomial Logistic Regression

ทดลองค่า `C = 0.001, 0.01, 0.1, 1 และ 10` ด้วย Rolling Backtest 4 รอบภายในชุด Train เท่านั้น โดยเลือกค่าที่มี Weighted MAE ต่ำที่สุด

ค่า `C` ต่ำหมายถึงการควบคุมความซับซ้อนของโมเดล (Regularization) ที่แรงขึ้น ช่วยลดโอกาสที่โมเดลจะจำรายละเอียดเฉพาะของข้อมูล Train มากเกินไป และไม่มีการใช้ Validation หรือ Test ในการเลือกค่า `C`


In [ ]:
tuning_results


In [ ]:
tuning_plot = tuning_results.copy()
tuning_plot["C_label"] = tuning_plot["C"].map(lambda value: f"{value:g}")
best_index = tuning_plot["rolling_weighted_mae"].idxmin()
bar_colors = [
    "#2E86AB" if index == best_index else "#B8C4CE"
    for index in tuning_plot.index
]

plt.figure(figsize=(9, 5))
sns.barplot(
    data=tuning_plot,
    x="C_label",
    y="rolling_weighted_mae",
    palette=bar_colors,
    hue="C_label",
    legend=False,
)
plt.title("Train-only Tuning: Rolling Weighted MAE")
plt.xlabel("Logistic Regression C")
plt.ylabel("Rolling Weighted MAE")
plt.tight_layout()
plt.show()


ค่า `C = 0.001` ให้ Rolling Weighted MAE ต่ำที่สุด จึงถูกเลือกใช้ฝึก Binomial Logistic Regression ใหม่ ทำให้ MAE ของ Logistic ลดจากประมาณ 4.91 เหลือ 4.25 จุดเปอร์เซ็นต์ อย่างไรก็ตาม Baseline ยังดีกว่าที่ประมาณ 3.93 จุดเปอร์เซ็นต์


## 14. เปรียบเทียบประสิทธิภาพบน Validation

Validation ครอบคลุมเดือนมกราคม–มีนาคม 2026 และไม่ได้ถูกใช้ฝึกโมเดล ค่าความคลาดเคลื่อนคำนวณแบบถ่วงน้ำหนักตามจำนวนสัญญาหมดอายุ


In [ ]:
validation_metrics = (
    pd.DataFrame(metrics["validation_2026_q1"])
    .T.reset_index(names="model")
)
validation_metrics[[
    "model", "weighted_mae", "weighted_rmse",
    "binomial_log_loss", "actual_overall_rate",
    "predicted_overall_rate", "contracts"
]]


In [ ]:
metric_plot = validation_metrics.melt(
    id_vars="model",
    value_vars=["weighted_mae", "weighted_rmse"],
    var_name="metric",
    value_name="value",
)

plt.figure(figsize=(9, 5))
sns.barplot(data=metric_plot, x="metric", y="value", hue="model")
plt.title("Validation Error Comparison")
plt.xlabel("Metric")
plt.ylabel("Error")
plt.tight_layout()
plt.show()


## 15. อัตราจริงเทียบกับอัตราที่โมเดลพยากรณ์

จุดที่อยู่ใกล้เส้นประหมายถึงอัตราที่พยากรณ์ใกล้เคียงอัตราจริง ขนาดจุดแสดงจำนวนสัญญาหมดอายุของโครงการ–เดือนนั้น


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
model_columns = {
    "Historical-rate Baseline": "baseline_predicted_rate",
    "Binomial Logistic Regression": "logistic_predicted_rate",
}

plot_min = min(
    validation_predictions["renewal_rate"].min(),
    validation_predictions[list(model_columns.values())].min().min(),
)
plot_max = max(
    validation_predictions["renewal_rate"].max(),
    validation_predictions[list(model_columns.values())].max().max(),
)

for axis, (model_name, prediction_column) in zip(axes, model_columns.items()):
    sns.scatterplot(
        data=validation_predictions,
        x="renewal_rate",
        y=prediction_column,
        size="expiring_contracts",
        sizes=(25, 230),
        alpha=0.7,
        ax=axis,
        legend=False,
    )
    axis.plot([plot_min, plot_max], [plot_min, plot_max], "--", color="red")
    axis.set_title(model_name)
    axis.set_xlabel("Actual Renewal Rate")
    axis.set_ylabel("Predicted Renewal Rate")

plt.tight_layout()
plt.show()


## 16. วิเคราะห์การกระจายของข้อผิดพลาด

Error คำนวณจาก `อัตราที่พยากรณ์ - อัตราจริง` ค่าบวกหมายถึงพยากรณ์สูงเกินไป และค่าลบหมายถึงพยากรณ์ต่ำเกินไป


In [ ]:
error_data = validation_predictions.copy()
error_data["baseline_error"] = (
    error_data["baseline_predicted_rate"] - error_data["renewal_rate"]
)
error_data["logistic_error"] = (
    error_data["logistic_predicted_rate"] - error_data["renewal_rate"]
)

error_long = error_data.melt(
    id_vars=["project_id", "expiry_month"],
    value_vars=["baseline_error", "logistic_error"],
    var_name="model",
    value_name="error",
)

plt.figure(figsize=(10, 5))
sns.histplot(data=error_long, x="error", hue="model", bins=18, kde=True)
plt.axvline(0, color="black", linestyle="--")
plt.title("Validation Error Distribution")
plt.xlabel("Predicted Rate - Actual Rate")
plt.tight_layout()
plt.show()


In [ ]:
largest_errors = validation_predictions.assign(
    baseline_absolute_error=lambda frame: abs(
        frame["baseline_predicted_rate"] - frame["renewal_rate"]
    ),
    logistic_absolute_error=lambda frame: abs(
        frame["logistic_predicted_rate"] - frame["renewal_rate"]
    ),
).sort_values("baseline_absolute_error", ascending=False)

largest_errors.head(10)


## 17. ตัวอย่างการนำโมเดลไปใช้งาน

เลือกโครงการ–เดือนใน Validation ที่มีจำนวนสัญญาหมดอายุมากที่สุด แล้วแสดงอัตราที่ Baseline และ Logistic Regression พยากรณ์เทียบกับอัตราจริง


In [ ]:
sample = validation_predictions.sort_values(
    "expiring_contracts", ascending=False
).iloc[0]

sample_features = renewal_rate_data.loc[
    renewal_rate_data["project_id"].eq(sample["project_id"])
    & renewal_rate_data["expiry_month"].eq(sample["expiry_month"]),
    RATE_MODEL_FEATURES,
]
model_probability = rate_model.predict_proba(sample_features)[:, 1][0]

example_result = pd.Series({
    "project_id": int(sample["project_id"]),
    "expiry_month": sample["expiry_month"].strftime("%Y-%m"),
    "expiring_contracts": int(sample["expiring_contracts"]),
    "actual_renewal_rate": sample["renewal_rate"],
    "historical_baseline": sample["baseline_predicted_rate"],
    "binomial_logistic": model_probability,
})
example_result


## 18. ประเมินโมเดลกับชุด Test ครั้งสุดท้าย

หลังจาก Tune และเลือก Historical-rate Baseline จาก Train และ Validation แล้ว จึงล็อกโมเดล ฝึกใหม่ด้วยข้อมูล Train+Validation และประเมินกับ Test ช่วงเมษายน–มิถุนายน 2026 เพียงครั้งเดียว

ชุด Test มี 48 โครงการ–เดือน รวม 5,228 สัญญาหมดอายุ และไม่ถูกใช้ในการ Tune หรือเลือกโมเดล


In [ ]:
test_metrics = (
    pd.DataFrame(metrics["test_2026_q2"])
    .T.reset_index(names="model")
)
test_metrics[[
    "model", "weighted_mae", "weighted_rmse",
    "binomial_log_loss", "actual_overall_rate",
    "predicted_overall_rate", "contracts"
]]


In [ ]:
test_metric_plot = test_metrics.melt(
    id_vars="model",
    value_vars=["weighted_mae", "weighted_rmse"],
    var_name="metric",
    value_name="value",
)

plt.figure(figsize=(9, 5))
sns.barplot(data=test_metric_plot, x="metric", y="value", hue="model")
plt.title("Final Test Error Comparison")
plt.xlabel("Metric")
plt.ylabel("Error")
plt.tight_layout()
plt.show()


บน Test นั้น Baseline มี MAE ประมาณ 3.49 จุดเปอร์เซ็นต์ ส่วน Logistic มี MAE ประมาณ 3.11 จุดเปอร์เซ็นต์ แม้ Logistic จะดีกว่าเล็กน้อยในช่วง Test นี้ แต่จะไม่เปลี่ยนโมเดลหลังเห็น Test เพราะจะทำให้ Test กลายเป็นข้อมูลเลือกโมเดล ควรรอข้อมูลอนาคตรอบใหม่เพื่อยืนยันผลอีกครั้ง


## 19. ตารางผลพยากรณ์สำหรับใช้ทางธุรกิจ

ตารางนี้ใช้โมเดลที่เลือกไว้ล่วงหน้า คือ Historical-rate Baseline เพื่อคำนวณอัตราที่คาดว่าจะต่อ จำนวนที่คาดว่าจะต่อ และจำนวนที่คาดว่าจะไม่ต่อในแต่ละโครงการ–เดือน

ระดับความเสี่ยง `low`, `medium`, `high` แบ่งจากจำนวนคาดว่าจะไม่ต่อเมื่อเทียบกับโครงการอื่นในเดือนเดียวกัน จึงใช้จัดลำดับภาระงานรักษาผู้เช่าในระดับโครงการ ไม่ใช่การทำนายรายบุคคล


In [ ]:
business_predictions.head(15)


In [ ]:
priority_projects = business_predictions.nlargest(
    12, "expected_not_renewed_contracts"
).copy()
priority_projects["project_month"] = (
    "Project " + priority_projects["project_id"].astype(str)
    + " | " + priority_projects["expiry_month"].dt.strftime("%Y-%m")
)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=priority_projects,
    x="expected_not_renewed_contracts",
    y="project_month",
    hue="risk_level",
    hue_order=["high", "medium", "low"],
)
plt.title("Projects with the Highest Expected Non-renewals")
plt.xlabel("Expected Non-renewed Contracts")
plt.ylabel("Project and Month")
plt.tight_layout()
plt.show()


## 20. สรุปผลสำหรับใช้เขียนรายงาน

โครงงานเปลี่ยนจากการทำนายการต่อสัญญารายสัญญาเป็นการพยากรณ์อัตราการต่อสัญญาระดับโครงการ–เดือน เนื่องจากข้อมูลไม่มีตัวระบุและพฤติกรรมผู้เช่าที่เพียงพอสำหรับการจำแนกรายบุคคล

ผลสำคัญ:
- ข้อมูลรวมมี 438 โครงการ–เดือน จาก 18 โครงการและ 37 เดือน
- Historical-rate Baseline มี Rolling Backtest MAE ประมาณ **3.9 จุดเปอร์เซ็นต์**
- การ Tune ภายใน Train เลือกค่า `C = 0.001` และลด Rolling MAE ของ Logistic จากประมาณ **4.9 เหลือ 4.3 จุดเปอร์เซ็นต์**
- บน Validation ไตรมาส 1/2026 Baseline มี MAE ประมาณ **4.3 จุดเปอร์เซ็นต์**
- Binomial Logistic Regression หลัง Tune มี Validation MAE ประมาณ **4.8 จุดเปอร์เซ็นต์** จึงยังไม่ชนะ Baseline
- บน Test ไตรมาส 2/2026 Baseline มี MAE ประมาณ **3.49 จุดเปอร์เซ็นต์** และ Logistic มี MAE ประมาณ **3.11 จุดเปอร์เซ็นต์**
- แม้ Logistic ดีกว่าเล็กน้อยบน Test แต่ไม่เปลี่ยนโมเดลหลังเห็นผล Test โดย Historical-rate Baseline ยังคงเป็นโมเดลที่เลือกไว้ล่วงหน้า
- ตารางผลเชิงธุรกิจช่วยประมาณจำนวนต่อ/ไม่ต่อและจัดลำดับความเสี่ยงระดับโครงการ–เดือน

ดังนั้น Historical-rate Baseline ควรเป็นวิธีหลักสำหรับประมาณอัตราและจำนวนสัญญาที่คาดว่าจะต่อในแต่ละโครงการ–เดือน ส่วน Logistic Regression ใช้เป็นโมเดลเปรียบเทียบทางสถิติ


## 21. เอกสารอ้างอิงของโมเดล

- McCullagh, P., & Nelder, J. A. (1989). *Generalized Linear Models* (2nd ed.). Chapman and Hall/CRC. https://doi.org/10.1007/978-1-4899-3242-6
- Tashman, L. J. (2000). Out-of-sample tests of forecasting accuracy: An analysis and review. *International Journal of Forecasting, 16*(4), 437–450. https://doi.org/10.1016/S0169-2070(00)00065-0
- การประเมินใช้การแบ่งข้อมูลตามเวลาและ Rolling Backtest เพื่อให้ลำดับการฝึกและตรวจสอบสอดคล้องกับการพยากรณ์จริง
